# Home Repair Trend Pipeline — v0.2
**Fixed:** Replaced pytrends with direct HTTP calls to Google Trends (avoids urllib3 compatibility bug and gives us full header control)

**Sources:** YouTube Data API v3 · Google Trends (direct requests)  

---
### If you just ran the diagnostic, wait 5-10 minutes before running the Trends cells.
Google issued a 429 rate limit. It clears quickly.

In [ ]:
# Cell 1 — Install dependencies from pinned requirements
import subprocess, sys, os

REQ_FILE = 'requirements.txt'
if os.path.exists(REQ_FILE):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ_FILE])
    print(f'Dependencies installed from {REQ_FILE}.')
else:
    # Fallback for ad-hoc runs outside the project folder
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'google-api-python-client',
        'pandas',
        'numpy',
        'requests',
        'python-dotenv'
    ])
    print('Dependencies installed (fallback package list).')

Dependencies installed.


In [ ]:
# Cell 2 — Configuration
import os
from dotenv import load_dotenv

load_dotenv()

YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY', 'PASTE_YOUR_KEY_HERE')

if YOUTUBE_API_KEY == 'PASTE_YOUR_KEY_HERE':
    print('⚠️  No YouTube API key found.')
else:
    print(f'✅ YouTube API key loaded: {YOUTUBE_API_KEY[:12]}...')

WEIGHT_YOUTUBE = 0.55
WEIGHT_TRENDS  = 0.45

YT_RESULTS_PER_TOPIC = 10
YT_ACTIVE_WINDOW_DAYS = 365

# Set True to score the full 10-topic list.
RUN_FULL_TOPIC_SET = True

# Seconds to sleep between Google Trends requests.
# Google allows roughly 1 request per 5-10 seconds from the same IP.
# If you get 429s, increase this.
TRENDS_SLEEP = 20

✅ YouTube API key loaded: AIzaSyAe35a8...


In [ ]:
# Cell 3 — Seed topics
SEED_TOPICS = [
    'garage door not closing',
    'water heater making noise',
    'bathroom exhaust fan replacement',
    'drywall hole repair',
    'leaking shower grout fix',
    'toilet running constantly',
    'exterior door weatherstripping',
    'HVAC filter replacement',
    'circuit breaker tripping',
    'roof shingle damage assessment',
]

SEED_TOPICS_SMALL = [
    'garage door not closing',
    'water heater making noise',
    'bathroom exhaust fan replacement',
]

SEED_TOPICS = SEED_TOPICS_BIG if RUN_FULL_TOPIC_SET else SEED_TOPICS_SMALL
mode = 'full (10 topics)' if RUN_FULL_TOPIC_SET else 'short (3 topics)'
print(f'Scoring {len(SEED_TOPICS)} topics | mode={mode}.')

Scoring 3 topics.


In [14]:
# Cell 4 — YouTube functions (stability + better velocity sampling)
from googleapiclient.discovery import build
from datetime import datetime, timedelta, timezone
import time
import pandas as pd

def build_youtube_client(api_key):
    return build('youtube', 'v3', developerKey=api_key)

def _search_video_ids(client, query, max_results, order, published_after=None):
    req_kwargs = {
        'q': query,
        'part': 'id,snippet',
        'type': 'video',
        'maxResults': max_results,
        'order': order,
        'relevanceLanguage': 'en',
    }
    if published_after is not None:
        req_kwargs['publishedAfter'] = published_after

    resp = client.search().list(**req_kwargs).execute()
    return [
        item['id']['videoId']
        for item in resp.get('items', [])
        if item.get('id', {}).get('videoId')
    ]

def fetch_youtube_videos(client, query, max_results=10, active_window_days=365):
    """
    Pull two slices and merge them:
      1) Relevance-ranked results (demand signal)
      2) Date-ranked recent uploads (momentum signal)
    """
    try:
        now_utc = datetime.now(timezone.utc)
        published_after = (now_utc - timedelta(days=active_window_days)).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

        ids_relevance = _search_video_ids(
            client, query, max_results=max_results, order='relevance'
        )
        ids_recent = _search_video_ids(
            client,
            query,
            max_results=max_results,
            order='date',
            published_after=published_after,
        )

        merged_ids = []
        for vid in ids_recent + ids_relevance:
            if vid not in merged_ids:
                merged_ids.append(vid)
        video_ids = merged_ids[: max_results * 2]

        if not video_ids:
            return []

        stats_resp = client.videos().list(
            id=','.join(video_ids),
            part='statistics,snippet',
        ).execute()

        videos = []
        for item in stats_resp.get('items', []):
            published_raw = item.get('snippet', {}).get('publishedAt', '')
            try:
                published_at = datetime.fromisoformat(published_raw.replace('Z', '+00:00'))
            except ValueError:
                published_at = None

            view_count = int(item.get('statistics', {}).get('viewCount', 0))
            videos.append({
                'video_id': item.get('id', ''),
                'title': item.get('snippet', {}).get('title', ''),
                'published_at': published_at,
                'view_count': view_count,
                'channel': item.get('snippet', {}).get('channelTitle', ''),
            })

        return videos

    except Exception as e:
        print(f'  YouTube error: {e}')
        return []

def compute_youtube_velocity(videos, active_window_days=365):
    """
    Returns a 0-1 velocity score using recency + VPD performance.
    Includes a fallback floor when no videos are inside the active window.
    """
    if not videos:
        return 0.0

    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(days=active_window_days)

    all_vpd = []
    recent_vpd = []
    valid_ages = []

    for v in videos:
        published_at = v.get('published_at')
        if published_at is None:
            continue

        age_days = max((now - published_at).days, 1)
        valid_ages.append(age_days)

        vpd = v.get('view_count', 0) / age_days
        all_vpd.append(vpd)
        if published_at >= cutoff:
            recent_vpd.append(vpd)

    if not all_vpd:
        return 0.0

    recency_fraction = len(recent_vpd) / len(all_vpd)

    if recent_vpd:
        median_all = sorted(all_vpd)[len(all_vpd) // 2]
        median_recent = sorted(recent_vpd)[len(recent_vpd) // 2]
        vpd_ratio = median_recent / (median_all + 1)
        vpd_signal = vpd_ratio / (vpd_ratio + 1)
    else:
        # Fallback: if no video is inside the strict window, still retain
        # a small momentum signal based on how recently the newest video was published.
        newest_age = min(valid_ages) if valid_ages else active_window_days * 5
        freshness = active_window_days / (active_window_days + newest_age)
        vpd_signal = max(0.15, min(freshness, 0.35))

    velocity = (recency_fraction * 0.65) + (vpd_signal * 0.35)

    print(
        f'         [yt debug] recency={recency_fraction:.2f} ({len(recent_vpd)}/{len(all_vpd)} recent)  '
        f'vpd_signal={vpd_signal:.2f}'
    )

    return round(min(velocity, 1.0), 4)

print('YouTube functions ready.')

YouTube functions ready.


In [15]:
# Cell 5 — Google Trends: direct HTTP with resilient retries
import requests
import json
import numpy as np
import random

TRENDS_SESSION = requests.Session()
TRENDS_SESSION.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Referer': 'https://trends.google.com/',
})

print('Warming up Google Trends session...')
_warmup = TRENDS_SESSION.get('https://trends.google.com/', timeout=15)
print(f'Warm-up status: {_warmup.status_code}')
time.sleep(2)

def _strip_xssi_prefix(raw_text):
    # Google Trends prepends a protective line like )]}'
    if '\n' in raw_text:
        return raw_text.split('\n', 1)[1]
    return raw_text

def _get_with_backoff(url, params, label, attempts=5, base_wait=8):
    """
    Retry on 429/5xx with exponential backoff + jitter.
    """
    for attempt in range(1, attempts + 1):
        try:
            resp = TRENDS_SESSION.get(url, params=params, timeout=20)
        except Exception as e:
            if attempt == attempts:
                raise RuntimeError(f'{label} request exception: {e}')
            wait_s = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
            print(f'  {label} network error (attempt {attempt}/{attempts}); waiting {wait_s:.1f}s...')
            time.sleep(wait_s)
            continue

        if resp.status_code == 200:
            return resp

        should_retry = resp.status_code == 429 or 500 <= resp.status_code < 600
        if not should_retry or attempt == attempts:
            return resp

        wait_s = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 2)
        print(
            f'  {label} HTTP {resp.status_code} (attempt {attempt}/{attempts}); '
            f'waiting {wait_s:.1f}s...'
        )
        time.sleep(wait_s)

    return None

def fetch_trends_data(keyword, timeframe='today 3-m', geo='US'):
    """
    Returns a list of (date_str, value) tuples, or [] on failure.
    Step 1: GET /trends/api/explore to get token + request payload.
    Step 2: GET /trends/api/widgetdata/multiline with token.
    """
    explore_url = 'https://trends.google.com/trends/api/explore'
    payload = {
        'hl': 'en-US',
        'tz': '360',
        'req': json.dumps({
            'comparisonItem': [{
                'keyword': keyword,
                'geo': geo,
                'time': timeframe,
            }],
            'category': 0,
            'property': '',
        }),
    }

    try:
        resp = _get_with_backoff(explore_url, payload, label='Trends explore')
        if resp is None or resp.status_code != 200:
            status = 'no response' if resp is None else f'HTTP {resp.status_code}'
            print(f'  Trends explore failed: {status}')
            return []

        data = json.loads(_strip_xssi_prefix(resp.text))

        widgets = data.get('widgets', [])
        token = None
        req_obj = None
        for w in widgets:
            if w.get('id') == 'TIMESERIES':
                token = w.get('token')
                req_obj = w.get('request')
                break

        if not token or not req_obj:
            print('  No TIMESERIES widget found in response')
            return []

    except Exception as e:
        print(f'  Trends explore error: {e}')
        return []

    multiline_url = 'https://trends.google.com/trends/api/widgetdata/multiline'
    multiline_params = {
        'hl': 'en-US',
        'tz': '360',
        'req': json.dumps(req_obj),
        'token': token,
        'user_type': 'USER_TYPE_SCRAPER',
    }

    try:
        resp2 = _get_with_backoff(multiline_url, multiline_params, label='Trends multiline')
        if resp2 is None or resp2.status_code != 200:
            status = 'no response' if resp2 is None else f'HTTP {resp2.status_code}'
            print(f'  Trends multiline failed: {status}')
            return []

        data2 = json.loads(_strip_xssi_prefix(resp2.text))

        points = []
        for row in data2.get('default', {}).get('timelineData', []):
            date_str = row.get('formattedAxisTime', row.get('formattedTime', ''))
            value = row.get('value', [0])[0]
            points.append((date_str, value))

        return points

    except Exception as e:
        print(f'  Trends multiline error: {e}')
        return []

def compute_trends_growth(keyword):
    """
    Fetch 90 days, compare last third vs first two thirds.
    Returns normalized 0-1 score.
    """
    points = fetch_trends_data(keyword, timeframe='today 3-m', geo='US')

    if len(points) < 8:
        print(f'  Not enough data points: {len(points)}')
        return 0.0

    values = np.array([p[1] for p in points], dtype=float)
    split = len(values) * 2 // 3
    baseline_avg = np.mean(values[:split]) + 0.01
    recent_avg = np.mean(values[split:])
    ratio = recent_avg / baseline_avg
    return round(min(ratio / (ratio + 1), 1.0), 4)

print('Trends functions ready (resilient retry mode).')

# Quick smoke test
print('\nSmoke test: fetching "garage door" trend...')
test_pts = fetch_trends_data('garage door', timeframe='today 3-m', geo='US')
if test_pts:
    print(f'✅ Got {len(test_pts)} data points. Last value: {test_pts[-1]}')
else:
    print('❌ Empty data after retries. Increase TRENDS_SLEEP or wait 10+ minutes before rerun.')

Warming up Google Trends session...
Warm-up status: 200
Trends functions ready (resilient retry mode).

Smoke test: fetching "garage door" trend...
✅ Got 90 data points. Last value: ('May 9', 57)


In [16]:
# Cell 6 — Run the full pipeline
results = []
yt_client = build_youtube_client(YOUTUBE_API_KEY)

for i, topic in enumerate(SEED_TOPICS):
    print(f'[{i+1}/{len(SEED_TOPICS)}] "{topic}"')

    # YouTube
    yt_videos = fetch_youtube_videos(
        yt_client,
        topic,
        max_results=YT_RESULTS_PER_TOPIC,
        active_window_days=YT_ACTIVE_WINDOW_DAYS,
    )
    yt_velocity = compute_youtube_velocity(
        yt_videos,
        active_window_days=YT_ACTIVE_WINDOW_DAYS,
    )
    print(f'       YT velocity:    {yt_velocity:.3f}  ({len(yt_videos)} videos)')

    # Trends — sleep between calls to stay under rate limit
    time.sleep(TRENDS_SLEEP)
    trends_growth = compute_trends_growth(topic)
    print(f'       Trends growth:  {trends_growth:.3f}')

    score = round(yt_velocity * WEIGHT_YOUTUBE + trends_growth * WEIGHT_TRENDS, 4)
    print(f'       Score:          {score:.3f}\n')

    results.append({
        'topic':            topic,
        'youtube_velocity': yt_velocity,
        'trends_growth':    trends_growth,
        'score':            score,
        'yt_videos':        len(yt_videos),
    })

print('Pipeline complete.')

[1/3] "garage door not closing"
         [yt debug] recency=0.55 (11/20 recent)  vpd_signal=0.23
       YT velocity:    0.440  (20 videos)
  Trends multiline HTTP 429 (attempt 1/5); waiting 8.3s...
       Trends growth:  0.714
       Score:          0.563

[2/3] "water heater making noise"
         [yt debug] recency=0.65 (13/20 recent)  vpd_signal=0.17
       YT velocity:    0.482  (20 videos)
  Trends multiline HTTP 429 (attempt 1/5); waiting 8.1s...
  Trends multiline HTTP 429 (attempt 2/5); waiting 17.0s...
       Trends growth:  0.551
       Score:          0.513

[3/3] "bathroom exhaust fan replacement"
         [yt debug] recency=0.55 (11/20 recent)  vpd_signal=0.32
       YT velocity:    0.469  (20 videos)
       Trends growth:  0.732
       Score:          0.587

Pipeline complete.


In [ ]:
# Cell 7 — Results
import json
from datetime import datetime, timezone

df = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
print('=== TREND SCORES ===')
print(df[['topic','youtube_velocity','trends_growth','score']].to_string())

output = {
    'meta': {
        'run_at':            datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'topic_count':       len(results),
        'weight_youtube':    WEIGHT_YOUTUBE,
        'weight_trends':     WEIGHT_TRENDS,
        'trends_sleep_s':    TRENDS_SLEEP,
        'yt_window_days':    YT_ACTIVE_WINDOW_DAYS,
        'yt_results_per_topic': YT_RESULTS_PER_TOPIC,
    },
    'results': df[['topic','youtube_velocity','trends_growth','score']].to_dict(orient='records'),
}

with open('trend_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('\nSaved: trend_results.json')
print(json.dumps(output['results'][:3], indent=2))

=== TREND SCORES ===
                              topic  youtube_velocity  trends_growth   score
0  bathroom exhaust fan replacement            0.4687         0.7321  0.5872
1           garage door not closing            0.4396         0.7141  0.5631
2         water heater making noise            0.4821         0.5507  0.5130

Saved: trend_results.json
[
  {
    "topic": "bathroom exhaust fan replacement",
    "youtube_velocity": 0.4687,
    "trends_growth": 0.7321,
    "score": 0.5872
  },
  {
    "topic": "garage door not closing",
    "youtube_velocity": 0.4396,
    "trends_growth": 0.7141,
    "score": 0.5631
  },
  {
    "topic": "water heater making noise",
    "youtube_velocity": 0.4821,
    "trends_growth": 0.5507,
    "score": 0.513
  }
]
